> **Versão 1.0.1 (25/11/2026) — exatamente a usada nos resultados finais da tese:** o termo "AUMENTADA"/"AUMENTADO" da flora é convertido para o nível 3 diretamente no `MAPA_CRUZES` (célula 6). Na v1.0.0, essa correção estava numa célula posterior que redefinia a função depois da harmonização e, por isso, não era aplicada. Saídas em `resultados_escore_clinico_2023_2024_flora_corrigida`.

# TESE UFMG — ÍNDICE LINEAR CALIBRADO & ESCORE DE PONTOS INTEIROS (OBJETIVO 4 / PreviTU)

**Versão corrigida, rápida (1–2 min na base inteira, sem travamento) e alinhada ao Objetivo 3:**
- **100% da base de desenvolvimento (2011–2022)** — sem subamostragem artificial (`LIMIT_SAMPLES` removido) e sem `joblib(loky)`/deadlock.
- **Motor `L-BFGS` (regressão logística penalizada L2)** — converge em fração de segundo por ajuste, com log de progresso em tempo real.
- **Mesmos 9 preditores fisiológicos do EAS (incluindo hemoglobina na fita) + 3 demográficos/assistenciais (`Idade_anos`, `Sexo`, `Origem`)** e dois braços automáticos: `EAS_SEM_GRAM` vs. `EAS_COM_GRAM` (4 marcadores tintoriais/morfológicos qualitativos do Gram).
- **Duas camadas clínicas interpretáveis:**
  - **Camada A (Índice Linear Contínuo Calibrado):** utiliza `log1p` em contagens assimétricas de sedimento (`SEDLEU`, `SEDHEM`), recalibração logística fora-da-dobra e desescalonamento algébrico exato para integração no **PreviTU**.
  - **Camada B (Escore Simplificado de Pontos Inteiros — método de Sullivan/Framingham):** ajustada diretamente sobre **estratos clínicos** (sem pontos artificiais por dado ausente), convertendo coeficientes $\beta$ em pontos inteiros (constante $B = 0{,}25$) com tabela direta `Pontuação Total -> Probabilidade (%)`.
- **Validação temporal congelada (2023–2024, pacientes novos)** com *bootstrap* **agrupado por `Paciente_ID`** (1.000 réplicas), utilidade operacional por 1.000 pedidos (*Rule-out* 98%, *Rule-out* 95%, Youden, *Rule-in* Espec ≥ 95%), Curvas de Decisão (DCA), gráficos 300 DPI e planilha Excel multi-abas.


## 0. Ambiente, importações e estilo gráfico


In [ ]:
import importlib.util, subprocess, sys
for pkg in ["openpyxl"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import json
import os
import re
import time
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.special import expit
from sklearn.calibration import calibration_curve
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from google.colab import drive

warnings.filterwarnings("ignore")

mpl.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 8.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


## 1. Configuração central (alinhada ao Objetivo 3: 2011–2022 vs. 2023–2024 pacientes novos)


In [ ]:
drive.mount("/content/drive", force_remount=False)

CONFIG = {
    "input_path": "/content/drive/MyDrive/Todos os arquivos GPT/UROT_UROC_GG_Base_Definitiva_2011_2024_2026-10-19.csv.gz",
    "sep": ";",
    "output_dir": "/content/drive/MyDrive/Todos os arquivos GPT/resultados_escore_clinico_2023_2024_flora_corrigida",
    "col_target": "Cultura_positiva",
    "col_group": "Paciente_ID",
    "col_year": "Ano",
    "col_gram_text": "Gram_texto",
    "train_max_year": 2022,
    "test_years": [2023, 2024],
    "exclude_seen_patients_in_test": True,
    "n_splits_cv": 5,
    "c_grid": [0.01, 0.05, 0.2, 1.0, 5.0],
    "n_boot_coef": 200,      # Bootstrap agrupado por paciente para IC 95% dos coeficientes (rápido com lbfgs)
    "n_boot_ci": 1000,       # Bootstrap agrupado por paciente no teste temporal 2023-2024
    "point_unit": 0.25,      # Constante B (log-odds) equivalente a 1 ponto inteiro na Camada B
    "random_state": 42,
}

if not os.path.exists(CONFIG["input_path"]) and CONFIG["input_path"].endswith(".gz"):
    alt_csv = CONFIG["input_path"][:-3]
    if os.path.exists(alt_csv):
        CONFIG["input_path"] = alt_csv

assert os.path.exists(CONFIG["input_path"]), f"Base não encontrada: {CONFIG['input_path']}"
OUT_DIR = Path(CONFIG["output_dir"])
FIG_DIR = OUT_DIR / "figuras"
FIG_DIR.mkdir(parents=True, exist_ok=True)
log(f"Base localizada: {CONFIG['input_path']}")
log(f"Saída: {OUT_DIR}")

HEMOGL_CANDIDATES = [
    "UROT__HEMOGL", "UROT__HEMOGLOBINA", "UROT__HEMOG", "UROT__HB", "UROT__SANGUE", "HEMOGL"
]

CATEGORICAL_FEATURES = ["Sexo", "Origem", "UROT__NITRIT"]
GRAM_STRUCTURED_FEATURES = ["gram_bacterias_presentes", "gram_bgn", "gram_cgp", "gram_levedura"]
FORBIDDEN_SUBSTRINGS = (
    "UROC", "CONTAGEM", "CULTURA", "REVISAO", "PENDEN", "CONFLITO",
    "ALERTA", "RECUPERAC", "FONTE", "LAUDO", "ISOLADO", "RESUL", "ATB", "MIC_"
)


## 2. Harmonização clínica (9 preditores fisiológicos do EAS + hemoglobina na fita + Gram morfológico)


In [ ]:
MAPA_CRUZES = {
    "NEGATIVO": 0.0, "NEGATIVA": 0.0, "NEG": 0.0, "AUSENTE": 0.0, "AUSENTES": 0.0, "NORMAL": 0.0, "NAO REAGENTE": 0.0,
    "TRACOS": 0.5, "TRAÇOS": 0.5, "TRACO": 0.5, "TRAÇO": 0.5, "VESTIGIOS": 0.5, "VESTÍGIOS": 0.5, "RARA": 0.5, "RARO": 0.5,
    "ESCASSA": 1.0, "ESCASSO": 1.0, "1+": 1.0, "+": 1.0, "LEVE": 1.0, "DISCRETA": 1.0, "DISCRETO": 1.0, "POSITIVO": 1.0,
    "2+": 2.0, "++": 2.0, "MODERADA": 2.0, "MODERADO": 2.0,
    "3+": 3.0, "+++": 3.0, "ACENTUADA": 3.0, "ACENTUADO": 3.0, "NUMEROSA": 3.0, "INTENSA": 3.0,
    "AUMENTADA": 3.0, "AUMENTADO": 3.0,  # CORREÇÃO 25/11/2026: flora "aumentada" = nível 3 (antes não era convertida)
    "4+": 4.0, "++++": 4.0, "MACICA": 4.0, "MACIÇA": 4.0,
}

def parse_densidade(val) -> float:
    if pd.isna(val):
        return np.nan
    s = str(val).strip().upper().replace(",", ".")
    m = re.search(r"(\d+(?:\.\d+)?)", s)
    if not m:
        return np.nan
    num = float(m.group(1))
    if 1000.0 <= num <= 1060.0:
        num /= 1000.0
    return num if 1.000 <= num <= 1.060 else np.nan

def parse_sedimento(val) -> float:
    if pd.isna(val):
        return np.nan
    s = str(val).strip().upper()
    if any(w in s for w in ["AUSENT", "NEGAT", "RARO", "RARA", "NENHUM"]):
        return 0.0
    if "NUMEROS" in s or "MACIC" in s or "INCONT" in s:
        return 100.0
    m = re.search(r"(\d+(?:[.,]\d+)?)", s)
    return float(m.group(1).replace(",", ".")) if m else np.nan

def parse_semiquantitativo(serie: pd.Series) -> pd.Series:
    s = serie.astype(str).str.strip().str.upper()
    num = pd.to_numeric(s.str.replace(",", ".", regex=False), errors="coerce")
    return num.fillna(s.map(MAPA_CRUZES))

def extrair_features_gram(serie_gram: pd.Series) -> pd.DataFrame:
    out = pd.DataFrame(index=serie_gram.index)
    s = serie_gram.astype(str).str.upper().str.strip()
    invalido = serie_gram.isna() | s.isin(["", "NAN", "NONE", "NAO REALIZADO", "NÃO REALIZADO"])
    negativo = s.str.contains(
        r"NAO FORAM OBSERVAD|NÃO FORAM OBSERVAD|AUSENCIA DE BACT|AUSÊNCIA DE BACT|NEGATIV", regex=True
    )
    out["gram_bacterias_presentes"] = np.where(invalido, np.nan, np.where(negativo, 0.0, 1.0))
    out["gram_bgn"] = np.where(invalido, np.nan, s.str.contains(r"GRAM.?NEGATIV|BGN|BACILO", regex=True).astype(float))
    out["gram_cgp"] = np.where(invalido, np.nan, s.str.contains(r"GRAM.?POSITIV|CGP|COCO", regex=True).astype(float))
    out["gram_levedura"] = np.where(invalido, np.nan, s.str.contains(r"LEVEDUR|FUNG|PSEUDO.?HIFA|CANDIDA", regex=True).astype(float))
    return out

def detectar_coluna_hemoglobina(df: pd.DataFrame) -> str:
    for cand in HEMOGL_CANDIDATES:
        if cand in df.columns:
            return cand
    for col in df.columns:
        u = col.upper()
        if "HEMOG" in u or ("UROT" in u and ("SANG" in u or u.endswith("__HB"))):
            return col
    urot_cols = [c for c in df.columns if "UROT" in c.upper()]
    raise KeyError(f"Coluna de hemoglobina na fita não encontrada. Colunas UROT: {urot_cols}")

def preparar_base(df_raw: pd.DataFrame, cfg: dict):
    df = df_raw.copy()
    df.columns = [c.strip() for c in df.columns]

    y = pd.to_numeric(df[cfg["col_target"]], errors="coerce")
    years = pd.to_numeric(df[cfg["col_year"]], errors="coerce")
    valid = y.isin([0, 1]) & df[cfg["col_group"]].notna() & years.notna()
    df = df.loc[valid].copy().reset_index(drop=True)
    y = y.loc[valid].astype(int).to_numpy()
    years = years.loc[valid].astype(int).to_numpy()
    groups = df[cfg["col_group"]].astype(str).str.strip().to_numpy()

    hemogl_col = detectar_coluna_hemoglobina(df)
    log(f"Coluna de hemoglobina na fita detectada: '{hemogl_col}'")

    X = pd.DataFrame(index=df.index)
    idade = pd.to_numeric(df["Idade_anos"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
    X["Idade_anos"] = idade.where((idade >= 0) & (idade <= 110), np.nan)

    grav = df["UROT__GRAVID"].apply(parse_densidade)
    X["UROT__GRAVID"] = grav
    X["UROT__GRAVID_DESVIO_QUAD"] = (grav - 1.015) ** 2

    ph = pd.to_numeric(df["UROT__PH"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
    X["UROT__PH"] = ph.where((ph >= 4.0) & (ph <= 9.5), np.nan)

    X["UROT__SEDLEU"] = df["UROT__SEDLEU"].apply(parse_sedimento)
    X["UROT__SEDHEM"] = df["UROT__SEDHEM"].apply(parse_sedimento)
    X["UROT__SEDLEU_LOG"] = np.log1p(X["UROT__SEDLEU"].clip(lower=0))
    X["UROT__SEDHEM_LOG"] = np.log1p(X["UROT__SEDHEM"].clip(lower=0))

    X["UROT__ESTERASE"] = parse_semiquantitativo(df["UROT__ESTERASE"])
    X["UROT__GLIC"] = parse_semiquantitativo(df["UROT__GLIC"])
    X["UROT__FLORA"] = parse_semiquantitativo(df["UROT__FLORA"])
    _fl = X["UROT__FLORA"].isna().mean(); log(f"Flora não convertida após a correção: {_fl:.2%} (esperado: ~0%)")
    X["UROT__HEMOGL"] = parse_semiquantitativo(df[hemogl_col])

    for col in CATEGORICAL_FEATURES:
        s = df[col].fillna("AUSENTE").astype(str).str.strip().str.upper()
        X[col] = s.replace({"": "AUSENTE", "NAN": "AUSENTE", "NONE": "AUSENTE", "NULL": "AUSENTE"})

    gram_df = extrair_features_gram(df[cfg["col_gram_text"]])
    for gcol in GRAM_STRUCTURED_FEATURES:
        X[gcol] = gram_df[gcol]

    for c in X.columns:
        if any(sub in c.upper() for sub in FORBIDDEN_SUBSTRINGS):
            raise ValueError(f"Coluna bloqueada por risco de vazamento: {c}")

    return X, y, groups, years, hemogl_col

log("Carregando base definitiva...")
df_raw = pd.read_csv(CONFIG["input_path"], sep=CONFIG["sep"], low_memory=False)
X_all, y_all, groups_all, years_all, hemogl_col_orig = preparar_base(df_raw, CONFIG)
del df_raw

train_mask = years_all <= CONFIG["train_max_year"]
X_tr = X_all.loc[train_mask].reset_index(drop=True)
y_tr = y_all[train_mask]
g_tr = groups_all[train_mask]

seen_patients = set(g_tr)
test_mask = np.isin(years_all, CONFIG["test_years"])
if CONFIG["exclude_seen_patients_in_test"]:
    test_mask = test_mask & (~np.isin(groups_all, list(seen_patients)))

X_ev = X_all.loc[test_mask].reset_index(drop=True)
y_ev = y_all[test_mask]
g_ev = groups_all[test_mask]

log(f"Desenvolvimento (<= {CONFIG['train_max_year']}): n={len(X_tr):,} ({len(seen_patients):,} pac, prev={y_tr.mean()*100:.2f}%)")
log(f"Teste Temporal ({CONFIG['test_years']}, pacientes novos): n={len(X_ev):,} ({len(np.unique(g_ev)):,} pac, prev={y_ev.mean()*100:.2f}%)")


## 3. Motor rápido (L-BFGS): Camada A (Índice Contínuo) e Camada B (Escore de Pontos em Estratos Clínicos)


In [ ]:
LABELS_CLINICOS = {
    "Idade_anos": "Idade (anos)",
    "UROT__GRAVID": "Densidade urinária (g/mL)",
    "UROT__GRAVID_DESVIO_QUAD": "Desvio quadrático da densidade (d − 1,015)²",
    "UROT__PH": "pH urinário",
    "UROT__SEDLEU_LOG": "Leucócitos no sedimento [ln(1 + células/campo)]",
    "UROT__SEDHEM_LOG": "Hemácias no sedimento [ln(1 + células/campo)]",
    "UROT__ESTERASE": "Esterase leucocitária (0–4+)",
    "UROT__GLIC": "Glicose urinária (0–4+)",
    "UROT__FLORA": "Flora bacteriana no sedimento (0–4+)",
    "UROT__HEMOGL": "Hemoglobina na fita reagente (0–4+)",
    "gram_bacterias_presentes": "Gram: Bactérias presentes",
    "gram_bgn": "Gram: Bacilos Gram-negativos (BGN)",
    "gram_cgp": "Gram: Cocos Gram-positivos (CGP)",
    "gram_levedura": "Gram: Leveduras / estruturas fúngicas",
}

def build_continuous_matrices(X_tr_df, X_ev_df, num_cols, cat_cols):
    imp_num = SimpleImputer(strategy="median")
    X_tr_num = imp_num.fit_transform(X_tr_df[num_cols])
    X_ev_num = imp_num.transform(X_ev_df[num_cols])

    scaler = StandardScaler()
    X_tr_z = scaler.fit_transform(X_tr_num)
    X_ev_z = scaler.transform(X_ev_num)

    ohe = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
    X_tr_ohe = ohe.fit_transform(X_tr_df[cat_cols])
    X_ev_ohe = ohe.transform(X_ev_df[cat_cols])
    ohe_cols = list(ohe.get_feature_names_out(cat_cols))

    X_tr_mat = np.hstack([X_tr_z, X_tr_ohe])
    X_ev_mat = np.hstack([X_ev_z, X_ev_ohe])

    meta = []
    for j, c in enumerate(num_cols):
        meta.append({
            "feature": c, "clinical_label": LABELS_CLINICOS.get(c, c), "kind": "numeric",
            "median": float(imp_num.statistics_[j]), "mean": float(scaler.mean_[j]), "scale": float(scaler.scale_[j]),
        })
    for c in ohe_cols:
        meta.append({
            "feature": c, "clinical_label": f"Categoria: {c}", "kind": "categorical",
            "median": 0.0, "mean": 0.0, "scale": 1.0,
        })
    return X_tr_mat, X_ev_mat, pd.DataFrame(meta)


def fit_l2_cv_and_recalibrate(X_tr_mat, y_tr, g_tr, X_ev_mat, c_grid, n_splits=5, seed=42):
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    best_c, best_pr, best_oof_eta = c_grid[0], -np.inf, None

    for c_val in c_grid:
        oof_eta = np.zeros(len(y_tr), dtype=float)
        for tr_i, va_i in cv.split(X_tr_mat, y_tr, g_tr):
            clf = LogisticRegression(penalty="l2", solver="lbfgs", C=c_val, max_iter=1000)
            clf.fit(X_tr_mat[tr_i], y_tr[tr_i])
            oof_eta[va_i] = clf.decision_function(X_tr_mat[va_i])
        pr = float(average_precision_score(y_tr, expit(oof_eta)))
        if pr > best_pr:
            best_pr, best_c, best_oof_eta = pr, c_val, oof_eta

    # Recalibração logística fora-da-dobra
    recal = LogisticRegression(penalty=None, solver="lbfgs", max_iter=500)
    recal.fit(best_oof_eta.reshape(-1, 1), y_tr)
    recal_a, recal_b = float(recal.intercept_[0]), float(recal.coef_[0][0])
    oof_prob_cal = expit(recal_a + recal_b * best_oof_eta)

    final_clf = LogisticRegression(penalty="l2", solver="lbfgs", C=best_c, max_iter=1000)
    final_clf.fit(X_tr_mat, y_tr)
    eta_tr_cal = recal_a + recal_b * final_clf.decision_function(X_tr_mat)
    eta_ev_cal = recal_a + recal_b * final_clf.decision_function(X_ev_mat)

    return {
        "best_c": best_c, "cv_pr_auc": best_pr,
        "recal_a": recal_a, "recal_b": recal_b,
        "final_clf": final_clf,
        "oof_prob_cal": oof_prob_cal,
        "prob_tr_cal": expit(eta_tr_cal),
        "prob_ev_cal": expit(eta_ev_cal),
        "eta_ev_cal": eta_ev_cal,
    }


def bootstrap_coefs_grouped(X_tr_mat, y_tr, g_tr, best_c, n_boot=200, seed=42):
    t0 = time.perf_counter()
    unique_g, inv_idx = np.unique(g_tr, return_inverse=True)
    g_indices = [np.where(inv_idx == i)[0] for i in range(len(unique_g))]
    n_g = len(unique_g)
    rng = np.random.default_rng(seed)

    coefs = []
    for b in range(1, n_boot + 1):
        sampled = rng.integers(0, n_g, size=n_g)
        idx = np.concatenate([g_indices[i] for i in sampled])
        yb = y_tr[idx]
        if yb.min() == yb.max():
            continue
        clf = LogisticRegression(penalty="l2", solver="lbfgs", C=best_c, max_iter=500)
        clf.fit(X_tr_mat[idx], yb)
        coefs.append(clf.coef_[0])
        if b % 50 == 0 or b == n_boot:
            log(f"    Bootstrap coeficientes (agrupado): {b}/{n_boot} ({time.perf_counter()-t0:.1f}s)")

    coefs = np.asarray(coefs)
    return np.percentile(coefs, 2.5, axis=0), np.percentile(coefs, 97.5, axis=0)


# --- CAMADA B: Discretização direta em estratos clínicos (método Sullivan / Framingham) ---
def discretizar_para_escore(df_in: pd.DataFrame, med_map: dict, incluir_gram: bool) -> pd.DataFrame:
    d = pd.DataFrame(index=df_in.index)

    # 1. Demografia e origem
    idade = df_in["Idade_anos"].fillna(med_map["Idade_anos"])
    d["Idade"] = pd.Categorical(
        np.select([idade < 12, idade < 40, idade < 65, idade < 80],
                  ["0_Crianca(<12a)", "1_AdultoJovem(12-39a)", "2_MeiaIdade(40-64a)", "3_Idoso(65-79a)"],
                  default="4_Longevo(>=80a)"),
        categories=["1_AdultoJovem(12-39a)", "0_Crianca(<12a)", "2_MeiaIdade(40-64a)", "3_Idoso(65-79a)", "4_Longevo(>=80a)"],
    )
    d["Sexo"] = pd.Categorical(df_in["Sexo"].where(df_in["Sexo"].isin(["F", "M"]), "M"), categories=["M", "F"])
    orig_clean = df_in["Origem"].where(df_in["Origem"].isin(["AMB", "PA", "INT", "BC", "HC"]), "AMB")
    orig_clean = orig_clean.replace({"BC": "AMB", "HC": "INT"})
    d["Origem"] = pd.Categorical(orig_clean, categories=["AMB", "PA", "INT"])

    # 2. 9 parâmetros fisiológicos do EAS em estratos clínicos
    est = df_in["UROT__ESTERASE"].fillna(med_map["UROT__ESTERASE"])
    d["Esterase"] = pd.Categorical(
        np.select([est <= 0.25, est <= 0.75, est <= 1.5, est <= 2.5],
                  ["0_Negativo", "1_Tracos", "2_1+", "3_2+"], default="4_3+ou4+"),
        categories=["0_Negativo", "1_Tracos", "2_1+", "3_2+", "4_3+ou4+"],
    )
    nit = df_in["UROT__NITRIT"].astype(str).str.upper()
    nit_pos = nit.str.contains("POS|1\+|2\+|3\+|REAG", regex=True) & (~nit.str.contains("NAO|NÃO|NEG", regex=True))
    d["Nitrito"] = pd.Categorical(np.where(nit_pos, "1_Positivo", "0_Negativo"), categories=["0_Negativo", "1_Positivo"])

    leu = df_in["UROT__SEDLEU"].fillna(med_map["UROT__SEDLEU"])
    d["Leucocitos"] = pd.Categorical(
        np.select([leu < 5, leu <= 15, leu <= 50], ["0_<5/campo", "1_5-15/campo", "2_16-50/campo"], default="3_>50/campo"),
        categories=["0_<5/campo", "1_5-15/campo", "2_16-50/campo", "3_>50/campo"],
    )
    hem = df_in["UROT__SEDHEM"].fillna(med_map["UROT__SEDHEM"])
    d["Hemacias"] = pd.Categorical(
        np.select([hem < 5, hem <= 20], ["0_<5/campo", "1_5-20/campo"], default="2_>20/campo"),
        categories=["0_<5/campo", "1_5-20/campo", "2_>20/campo"],
    )
    hb = df_in["UROT__HEMOGL"].fillna(med_map["UROT__HEMOGL"])
    d["Hemoglobina_fita"] = pd.Categorical(
        np.select([hb <= 0.25, hb <= 1.5], ["0_Negativo", "1_Tracos_a_1+"], default="2_>=2+"),
        categories=["0_Negativo", "1_Tracos_a_1+", "2_>=2+"],
    )
    flora = df_in["UROT__FLORA"].fillna(med_map["UROT__FLORA"])
    d["Flora"] = pd.Categorical(
        np.select([flora <= 0.25, flora <= 1.25, flora <= 2.25], ["0_Ausente", "1_Discreta", "2_Moderada"], default="3_Acentuada"),
        categories=["0_Ausente", "1_Discreta", "2_Moderada", "3_Acentuada"],
    )
    grav = df_in["UROT__GRAVID"].fillna(med_map["UROT__GRAVID"])
    d["Densidade"] = pd.Categorical(
        np.select([grav < 1.010, grav <= 1.020], ["1_Diluida(<1.010)", "0_Normal(1.010-1.020)"], default="2_Concentrada(>1.020)"),
        categories=["0_Normal(1.010-1.020)", "1_Diluida(<1.010)", "2_Concentrada(>1.020)"],
    )
    ph = df_in["UROT__PH"].fillna(med_map["UROT__PH"])
    d["pH"] = pd.Categorical(
        np.select([ph <= 5.5, ph <= 6.5], ["0_Acido(<=5.5)", "1_Neutro(6.0-6.5)"], default="2_Alcalino(>=7.0)"),
        categories=["1_Neutro(6.0-6.5)", "0_Acido(<=5.5)", "2_Alcalino(>=7.0)"],
    )
    glic = df_in["UROT__GLIC"].fillna(med_map["UROT__GLIC"])
    d["Glicose"] = pd.Categorical(np.where(glic <= 0.5, "0_Ausente", "1_Presente(>=1+)"), categories=["0_Ausente", "1_Presente(>=1+)"])

    # 3. Marcadores tintoriais/morfológicos do Gram (apenas no braço com Gram)
    if incluir_gram:
        for gcol, label in [
            ("gram_bacterias_presentes", "Gram_Bacterias"),
            ("gram_bgn", "Gram_BGN"),
            ("gram_cgp", "Gram_CGP"),
            ("gram_levedura", "Gram_Levedura"),
        ]:
            v = df_in[gcol].fillna(0.0)
            d[label] = pd.Categorical(np.where(v >= 0.5, "1_Presente", "0_Ausente"), categories=["0_Ausente", "1_Presente"])

    return d


def construir_escore_sullivan(X_tr_df, y_tr, g_tr, X_ev_df, y_ev, incluir_gram, c_grid, point_unit=0.25, n_splits=5, seed=42):
    med_map = {
        c: float(np.nanmedian(X_tr_df[c]))
        for c in ["Idade_anos", "UROT__ESTERASE", "UROT__SEDLEU", "UROT__SEDHEM", "UROT__HEMOGL", "UROT__FLORA", "UROT__GRAVID", "UROT__PH", "UROT__GLIC"]
    }
    D_tr = discretizar_para_escore(X_tr_df, med_map, incluir_gram)
    D_ev = discretizar_para_escore(X_ev_df, med_map, incluir_gram)

    cols_cat = list(D_tr.columns)
    ohe = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
    M_tr = ohe.fit_transform(D_tr[cols_cat])
    M_ev = ohe.transform(D_ev[cols_cat])
    feat_names = list(ohe.get_feature_names_out(cols_cat))

    fit_res = fit_l2_cv_and_recalibrate(M_tr, y_tr, g_tr, M_ev, c_grid, n_splits=n_splits, seed=seed)
    betas_cal = fit_res["recal_b"] * fit_res["final_clf"].coef_[0]
    intercept_cal = fit_res["recal_a"] + fit_res["recal_b"] * float(fit_res["final_clf"].intercept_[0])

    # Converte betas calibrados em pontos inteiros (constante B = point_unit)
    pts_int_vec = np.rint(betas_cal / point_unit).astype(int)
    score_tr = (M_tr @ pts_int_vec).astype(int)
    score_ev = (M_ev @ pts_int_vec).astype(int)

    # Calibração logística direta da pontuação total no desenvolvimento (2011–2022)
    cal_score = LogisticRegression(penalty=None, solver="lbfgs", max_iter=500)
    cal_score.fit(score_tr.reshape(-1, 1), y_tr)
    score_a = float(cal_score.intercept_[0])
    score_b = float(cal_score.coef_[0][0])

    prob_tr_score = expit(score_a + score_b * score_tr)
    prob_ev_score = expit(score_a + score_b * score_ev)

    # Tabela limpa de pontos por estrato (incluindo categoria de referência = 0 pontos)
    pts_rows = []
    for col_i, col_name in enumerate(cols_cat):
        cats = list(ohe.categories_[col_i])
        pts_rows.append({
            "variavel": col_name, "estrato": cats[0], "referencia": True,
            "beta_calibrado": 0.0, "or_ajustado": 1.0, "pontos_exatos": 0.0, "pontos_inteiros": 0,
        })
        for cat_val in cats[1:]:
            fname = f"{col_name}_{cat_val}"
            j = feat_names.index(fname)
            b_val = float(betas_cal[j])
            pts_rows.append({
                "variavel": col_name, "estrato": cat_val, "referencia": False,
                "beta_calibrado": round(b_val, 4), "or_ajustado": round(float(np.exp(b_val)), 3),
                "pontos_exatos": round(b_val / point_unit, 2), "pontos_inteiros": int(pts_int_vec[j]),
            })
    points_df = pd.DataFrame(pts_rows)

    # Tabela lookup Pontuação Total -> Probabilidade (%)
    min_s, max_s = int(min(score_tr.min(), score_ev.min())), int(max(score_tr.max(), score_ev.max()))
    lookup_rows = []
    for s_val in range(min_s, max_s + 1):
        m_tr, m_ev = score_tr == s_val, score_ev == s_val
        if m_tr.sum() == 0 and m_ev.sum() == 0:
            continue
        p_pred = float(expit(score_a + score_b * s_val))
        lookup_rows.append({
            "pontuacao_total": s_val,
            "probabilidade_predita_pct": round(p_pred * 100.0, 2),
            "n_desenvolvimento": int(m_tr.sum()),
            "positividade_obs_dev_pct": round(float(y_tr[m_tr].mean() * 100.0), 2) if m_tr.sum() > 0 else np.nan,
            "n_teste_2023_2024": int(m_ev.sum()),
            "positividade_obs_teste_pct": round(float(y_ev[m_ev].mean() * 100.0), 2) if m_ev.sum() > 0 else np.nan,
        })
    lookup_df = pd.DataFrame(lookup_rows)

    return {
        "points_df": points_df, "lookup_df": lookup_df,
        "score_tr": score_tr, "score_ev": score_ev,
        "prob_tr_score": prob_tr_score, "prob_ev_score": prob_ev_score,
        "score_cal_intercept": score_a, "score_cal_slope": score_b,
        "best_c": fit_res["best_c"],
    }


## 4. Avaliação operacional (por 1.000 pedidos) e bootstrap pareado agrupado por paciente


In [ ]:
def definir_limiares_desenvolvimento(y_tr, prob_tr) -> dict:
    fpr, tpr, ths = roc_curve(y_tr, prob_tr)
    idx_98 = np.where(tpr >= 0.98)[0]
    th_ro98 = float(ths[idx_98[np.argmax(1 - fpr[idx_98])]]) if len(idx_98) > 0 else float(ths[0])

    idx_95 = np.where(tpr >= 0.95)[0]
    th_ro95 = float(ths[idx_95[np.argmax(1 - fpr[idx_95])]]) if len(idx_95) > 0 else float(ths[0])

    th_youden = float(ths[int(np.argmax(tpr - fpr))])

    spec = 1.0 - fpr
    idx_sp95 = np.where(spec >= 0.95)[0]
    th_ri95 = float(ths[idx_sp95[np.argmax(tpr[idx_sp95])]]) if len(idx_sp95) > 0 else float(ths[-1])

    return {
        "Rule-out (Sens >= 98%)": th_ro98,
        "Rule-out (Sens >= 95%)": th_ro95,
        "Índice de Youden (Max J)": th_youden,
        "Rule-in (Espec >= 95%)": th_ri95,
    }


def metricas_operacionais_por_1000(y_true, p_pred, limiares_dict) -> pd.DataFrame:
    rows = []
    n = len(y_true)
    for zona, th in limiares_dict.items():
        y_hat = (p_pred >= th).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0, 1]).ravel()
        sens = tp / max(1, tp + fn)
        spec = tn / max(1, tn + fp)
        ppv = tp / max(1, tp + fp)
        npv = tn / max(1, tn + fn)
        rows.append({
            "zona_operacional": zona,
            "limiar_prob": round(th, 4),
            "sensibilidade": round(sens, 4),
            "especificidade": round(spec, 4),
            "vpp": round(ppv, 4),
            "vpn": round(npv, 4),
            "culturas_evitaveis_pct": round(((tn + fn) / n) * 100.0, 2),
            "culturas_evitaveis_por_1000": round(((tn + fn) / n) * 1000.0, 1),
            "positivas_nao_sinalizadas_por_1000": round((fn / n) * 1000.0, 1),
        })
    return pd.DataFrame(rows)


def paired_grouped_bootstrap_eval(y_true, p_a, p_b, groups, n_boot=1000, seed=42) -> dict:
    rng = np.random.default_rng(seed)
    unique_g, inv_idx = np.unique(groups, return_inverse=True)
    g_indices = [np.where(inv_idx == i)[0] for i in range(len(unique_g))]
    n_g = len(unique_g)

    def _calib(yv, pv):
        pc = np.clip(pv, 1e-6, 1.0 - 1e-6)
        lp = np.log(pc / (1.0 - pc)).reshape(-1, 1)
        lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=500)
        lr.fit(lp, yv)
        return float(lr.intercept_[0]), float(lr.coef_[0][0])

    auc_a, auc_b = float(roc_auc_score(y_true, p_a)), float(roc_auc_score(y_true, p_b))
    pr_a, pr_b = float(average_precision_score(y_true, p_a)), float(average_precision_score(y_true, p_b))
    br_a, br_b = float(brier_score_loss(y_true, p_a)), float(brier_score_loss(y_true, p_b))
    ic_a, sl_a = _calib(y_true, p_a)
    ic_b, sl_b = _calib(y_true, p_b)

    ba, bb, da, bpa, bpb, dp, db = [], [], [], [], [], [], []
    for _ in range(n_boot):
        sampled = rng.integers(0, n_g, size=n_g)
        idx = np.concatenate([g_indices[i] for i in sampled])
        yb = y_true[idx]
        if yb.min() == yb.max():
            continue
        aa, ab = roc_auc_score(yb, p_a[idx]), roc_auc_score(yb, p_b[idx])
        pa, pb = average_precision_score(yb, p_a[idx]), average_precision_score(yb, p_b[idx])
        bra, brb = brier_score_loss(yb, p_a[idx]), brier_score_loss(yb, p_b[idx])
        ba.append(aa); bb.append(ab); da.append(aa - ab)
        bpa.append(pa); bpb.append(pb); dp.append(pa - pb)
        db.append(bra - brb)

    def _p_two(arr):
        arr = np.asarray(arr, dtype=float)
        n_v = len(arr)
        return float(min(1.0, 2.0 * min((np.sum(arr <= 0) + 1) / (n_v + 1), (np.sum(arr >= 0) + 1) / (n_v + 1))))

    def _ci(arr):
        return float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5))

    return {
        "auc_a": auc_a, "auc_a_ci": _ci(ba),
        "auc_b": auc_b, "auc_b_ci": _ci(bb),
        "delta_auc": auc_a - auc_b, "delta_auc_ci": _ci(da), "delta_auc_p": _p_two(da),
        "pr_a": pr_a, "pr_a_ci": _ci(bpa),
        "pr_b": pr_b, "pr_b_ci": _ci(bpb),
        "delta_pr": pr_a - pr_b, "delta_pr_ci": _ci(dp), "delta_pr_p": _p_two(dp),
        "brier_a": br_a, "brier_b": br_b, "delta_brier": br_a - br_b, "delta_brier_p": _p_two(db),
        "ic_a": ic_a, "sl_a": sl_a, "ic_b": ic_b, "sl_b": sl_b,
    }


## 5. Execução dos dois braços (`EAS_SEM_GRAM` e `EAS_COM_GRAM`), tabelas e gráficos 300 DPI


In [ ]:
NUM_CONT_BASE = [
    "Idade_anos", "UROT__GRAVID", "UROT__GRAVID_DESVIO_QUAD", "UROT__PH",
    "UROT__SEDLEU_LOG", "UROT__SEDHEM_LOG", "UROT__ESTERASE", "UROT__GLIC",
    "UROT__FLORA", "UROT__HEMOGL",
]

ARMS = [
    {"name": "EAS_SEM_GRAM", "label": "EAS Fisiológico + Demografia (Sem Gram)", "incluir_gram": False},
    {"name": "EAS_COM_GRAM", "label": "EAS Fisiológico + Demografia + Gram Morfológico", "incluir_gram": True},
]

resultados = {}
resumo_rows = []
op_all_tables = []
coef_all_tables = []
pts_all_tables = []
lookup_all_tables = []

for arm in ARMS:
    arm_name = arm["name"]
    incluir_gram = arm["incluir_gram"]
    num_cols = NUM_CONT_BASE + (GRAM_STRUCTURED_FEATURES if incluir_gram else [])

    log("=" * 76)
    log(f"INICIANDO BRAÇO: {arm_name} ({arm['label']})")
    arm_dir = OUT_DIR / arm_name
    arm_dir.mkdir(parents=True, exist_ok=True)

    # --- CAMADA A: Índice Linear Contínuo Calibrado ---
    X_tr_mat, X_ev_mat, meta_df = build_continuous_matrices(X_tr, X_ev, num_cols, CATEGORICAL_FEATURES)
    fit_a = fit_l2_cv_and_recalibrate(
        X_tr_mat, y_tr, g_tr, X_ev_mat,
        c_grid=CONFIG["c_grid"], n_splits=CONFIG["n_splits_cv"], seed=CONFIG["random_state"],
    )
    log(f"  Camada A (Índice Contínuo): best_C={fit_a['best_c']} | OOF PR-AUC={fit_a['cv_pr_auc']:.4f} | Recalib: a={fit_a['recal_a']:.3f}, b={fit_a['recal_b']:.3f}")

    ci_lo_z, ci_hi_z = bootstrap_coefs_grouped(
        X_tr_mat, y_tr, g_tr, best_c=fit_a["best_c"],
        n_boot=CONFIG["n_boot_coef"], seed=CONFIG["random_state"],
    )

    beta_z_cal = fit_a["recal_b"] * fit_a["final_clf"].coef_[0]
    intercept_orig = fit_a["recal_a"] + fit_a["recal_b"] * float(fit_a["final_clf"].intercept_[0])
    beta_orig = np.zeros(len(meta_df), dtype=float)
    for j, row in meta_df.iterrows():
        if row["kind"] == "numeric":
            b_o = beta_z_cal[j] / row["scale"]
            beta_orig[j] = b_o
            intercept_orig -= b_o * row["mean"]
        else:
            beta_orig[j] = beta_z_cal[j]

    # Verificação algébrica exata do desescalonamento
    X_ev_orig = X_ev_mat * meta_df["scale"].values + meta_df["mean"].values
    max_diff = float(np.max(np.abs(fit_a["eta_ev_cal"] - (intercept_orig + X_ev_orig @ beta_orig))))
    assert max_diff < 1e-8, f"Erro no desescalonamento: {max_diff}"

    coef_df = meta_df.copy()
    coef_df["arm"] = arm_name
    coef_df["beta_std_cal"] = beta_z_cal
    coef_df["beta_std_ci95_low"] = fit_a["recal_b"] * ci_lo_z
    coef_df["beta_std_ci95_high"] = fit_a["recal_b"] * ci_hi_z
    coef_df["beta_orig_cal"] = beta_orig
    coef_df["or_orig_cal"] = np.exp(beta_orig)
    coef_df.to_csv(arm_dir / "coeficientes_indice_continuo.csv", index=False)
    coef_all_tables.append(coef_df)

    # --- CAMADA B: Escore Simplificado de Pontos Inteiros (Sullivan / Framingham) ---
    fit_b = construir_escore_sullivan(
        X_tr, y_tr, g_tr, X_ev, y_ev, incluir_gram=incluir_gram,
        c_grid=CONFIG["c_grid"], point_unit=CONFIG["point_unit"],
        n_splits=CONFIG["n_splits_cv"], seed=CONFIG["random_state"],
    )
    pts_df = fit_b["points_df"].copy()
    pts_df.insert(0, "arm", arm_name)
    lookup_df = fit_b["lookup_df"].copy()
    lookup_df.insert(0, "arm", arm_name)

    pts_df.to_csv(arm_dir / "escore_pontos_por_estrato.csv", index=False)
    lookup_df.to_csv(arm_dir / "escore_tabela_risco_total.csv", index=False)
    pts_all_tables.append(pts_df)
    lookup_all_tables.append(lookup_df)

    # --- Limiares no desenvolvimento e avaliação pareada agrupada em 2023–2024 ---
    lim_a = definir_limiares_desenvolvimento(y_tr, fit_a["oof_prob_cal"])
    lim_b = definir_limiares_desenvolvimento(y_tr, fit_b["prob_tr_score"])

    op_a = metricas_operacionais_por_1000(y_ev, fit_a["prob_ev_cal"], lim_a)
    op_a.insert(0, "camada", "Camada A: Índice Contínuo")
    op_a.insert(0, "arm", arm_name)
    op_b = metricas_operacionais_por_1000(y_ev, fit_b["prob_ev_score"], lim_b)
    op_b.insert(0, "camada", "Camada B: Escore de Pontos")
    op_b.insert(0, "arm", arm_name)
    op_comb = pd.concat([op_a, op_b], ignore_index=True)
    op_comb.to_csv(arm_dir / "metricas_operacionais_por_1000.csv", index=False)
    op_all_tables.append(op_comb)

    comp = paired_grouped_bootstrap_eval(
        y_ev, fit_a["prob_ev_cal"], fit_b["prob_ev_score"], g_ev,
        n_boot=CONFIG["n_boot_ci"], seed=CONFIG["random_state"],
    )
    log(
        f"  Teste 2023–2024 ({arm_name}) | Contínuo AUROC={comp['auc_a']:.4f} vs. Escore AUROC={comp['auc_b']:.4f} "
        f"(Δ={comp['delta_auc']:+.4f}, p={comp['delta_auc_p']:.4f}) | Brier: {comp['brier_a']:.4f} vs. {comp['brier_b']:.4f}"
    )

    resumo_rows.append({
        "arm": arm_name,
        "auroc_continuo": round(comp["auc_a"], 4),
        "auroc_continuo_ic95": f"{comp['auc_a_ci'][0]:.4f}–{comp['auc_a_ci'][1]:.4f}",
        "auroc_escore": round(comp["auc_b"], 4),
        "auroc_escore_ic95": f"{comp['auc_b_ci'][0]:.4f}–{comp['auc_b_ci'][1]:.4f}",
        "delta_auroc_cont_menos_escore": round(comp["delta_auc"], 4),
        "delta_auroc_p": round(comp["delta_auc_p"], 4),
        "prauc_continuo": round(comp["pr_a"], 4),
        "prauc_escore": round(comp["pr_b"], 4),
        "delta_prauc": round(comp["delta_pr"], 4),
        "brier_continuo": round(comp["brier_a"], 4),
        "brier_escore": round(comp["brier_b"], 4),
        "calib_intercept_continuo": round(comp["ic_a"], 4),
        "calib_slope_continuo": round(comp["sl_a"], 4),
        "calib_intercept_escore": round(comp["ic_b"], 4),
        "calib_slope_escore": round(comp["sl_b"], 4),
    })

    # Exporta equação fechada e JSON congelado para o PreviTU
    eq_lines = [
        f"=== EQUAÇÃO FECHADA DO ÍNDICE LINEAR CALIBRADO ({arm_name}) ===",
        f"Erro máximo de verificação algébrica: {max_diff:.2e}",
        f"eta_cal = {intercept_orig:+.6f}",
    ]
    for _, r in coef_df.iterrows():
        eq_lines.append(f"        {r['beta_orig_cal']:+.6f} * [{r['feature']}]  # {r['clinical_label']}")
    eq_lines.append("P(Positivo | x) = 1 / (1 + exp(-eta_cal))")
    (arm_dir / "equacao_indice_linear.txt").write_text("\n".join(eq_lines), encoding="utf-8")

    payload = {
        "arm": arm_name,
        "hemoglobina_col_origem": hemogl_col_orig,
        "intercept_orig": float(intercept_orig),
        "coeficientes_originais": dict(zip(coef_df["feature"], coef_df["beta_orig_cal"])),
        "medianas_imputacao": dict(zip(coef_df["feature"], coef_df["median"])),
        "limiares_continuo": lim_a,
        "escore_pontos": {
            "point_unit": CONFIG["point_unit"],
            "score_cal_intercept": fit_b["score_cal_intercept"],
            "score_cal_slope": fit_b["score_cal_slope"],
            "limiares_escore": lim_b,
            "tabela_pontos": pts_df.to_dict(orient="records"),
            "tabela_risco_total": lookup_df.to_dict(orient="records"),
        },
    }
    (arm_dir / "modelo_escore_congelado.json").write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

    resultados[arm_name] = {
        "prob_ev_cont": fit_a["prob_ev_cal"],
        "prob_ev_score": fit_b["prob_ev_score"],
        "lookup_df": lookup_df,
        "pts_df": pts_df,
    }

resumo_df = pd.DataFrame(resumo_rows)
op_full_df = pd.concat(op_all_tables, ignore_index=True)

# Comparação pareada Com Gram vs. Sem Gram para Camada A e Camada B
gram_comp_rows = []
for camada_key, camada_label in [("prob_ev_cont", "Camada A: Índice Contínuo"), ("prob_ev_score", "Camada B: Escore de Pontos")]:
    gc = paired_grouped_bootstrap_eval(
        y_ev, resultados["EAS_COM_GRAM"][camada_key], resultados["EAS_SEM_GRAM"][camada_key],
        g_ev, n_boot=CONFIG["n_boot_ci"], seed=CONFIG["random_state"],
    )
    gram_comp_rows.append({
        "camada": camada_label,
        "auroc_com_gram": round(gc["auc_a"], 4),
        "auroc_sem_gram": round(gc["auc_b"], 4),
        "delta_auroc": round(gc["delta_auc"], 4),
        "delta_auroc_ic95": f"{gc['delta_auc_ci'][0]:+.4f} a {gc['delta_auc_ci'][1]:+.4f}",
        "delta_auroc_p": round(gc["delta_auc_p"], 4),
        "delta_prauc": round(gc["delta_pr"], 4),
        "delta_prauc_p": round(gc["delta_pr_p"], 4),
    })
gram_comp_df = pd.DataFrame(gram_comp_rows)

# Exibe tabelas formatadas
print("\n=== TABELA 1: ÍNDICE CONTÍNUO VS. ESCORE DE PONTOS (TESTE 2023–2024, PACIENTES NOVOS) ===")
display(resumo_df)
print("\n=== TABELA 2: GANHO INCREMENTAL DO GRAM NAS DUAS CAMADAS ===")
display(gram_comp_df)
print("\n=== TABELA 3: IMPACTO OPERACIONAL POR 1.000 PEDIDOS ===")
display(op_full_df)
for arm in ARMS:
    print(f"\n=== TABELA DE PONTOS INTEIROS ({arm['name']}) ===")
    display(resultados[arm["name"]]["pts_df"])

# Exporta planilha Excel consolidada
xlsx_path = OUT_DIR / "Tabelas_Escore_Clinico_2023_2024.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    resumo_df.to_excel(writer, sheet_name="Resumo_CamadaA_vs_B", index=False)
    gram_comp_df.to_excel(writer, sheet_name="Ganho_Gram_Camadas", index=False)
    op_full_df.to_excel(writer, sheet_name="Operacional_Por_1000", index=False)
    pd.concat(coef_all_tables, ignore_index=True).to_excel(writer, sheet_name="Coeficientes_Camada_A", index=False)
    pd.concat(pts_all_tables, ignore_index=True).to_excel(writer, sheet_name="Pontos_Camada_B", index=False)
    pd.concat(lookup_all_tables, ignore_index=True).to_excel(writer, sheet_name="Tabela_Risco_Total", index=False)

# --- GRÁFICOS ACADÊMICOS (300 DPI) ---
# 1. ROC e PR (Camada A vs. Camada B nos dois braços)
fig, axes = plt.subplots(2, 2, figsize=(12.5, 10))
prev = float(np.mean(y_ev))
for r_i, arm in enumerate(ARMS):
    an = arm["name"]
    pc, ps = resultados[an]["prob_ev_cont"], resultados[an]["prob_ev_score"]
    ax_r, ax_p = axes[r_i, 0], axes[r_i, 1]

    for lbl, p, ls, col in [
        ("Camada A: Índice Contínuo", pc, "-", "#1d3557"),
        ("Camada B: Escore de Pontos", ps, "--", "#e63946"),
    ]:
        fpr, tpr, _ = roc_curve(y_ev, p)
        ax_r.plot(fpr, tpr, ls=ls, lw=2.2, color=col, label=f"{lbl} (AUROC={roc_auc_score(y_ev, p):.3f})")
        prec, rec, _ = precision_recall_curve(y_ev, p)
        ax_p.plot(rec, prec, ls=ls, lw=2.2, color=col, label=f"{lbl} (PR-AUC={average_precision_score(y_ev, p):.3f})")

    ax_r.plot([0, 1], [0, 1], ":", color="#888888")
    ax_r.set_title(f"Curva ROC — {arm['label']}")
    ax_r.set_xlabel("1 − Especificidade"); ax_r.set_ylabel("Sensibilidade")
    ax_r.legend(loc="lower right", frameon=True, facecolor="white"); ax_r.grid(alpha=0.25)

    ax_p.axhline(prev, ls=":", color="#888888", label=f"Prevalência ({prev:.3f})")
    ax_p.set_title(f"Curva Precisão–Revocação — {arm['label']}")
    ax_p.set_xlabel("Sensibilidade"); ax_p.set_ylabel("Precisão (VPP)")
    ax_p.legend(loc="lower left", frameon=True, facecolor="white"); ax_p.grid(alpha=0.25)

fig.suptitle("Índice Linear Contínuo vs. Escore de Pontos Inteiros (Teste Temporal 2023–2024)", fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / "01_ROC_PR_Indice_vs_Escore_2023_2024.png", bbox_inches="tight")
plt.show()

# 2. Calibração + Curva de Decisão (DCA)
fig, axes = plt.subplots(2, 2, figsize=(12.5, 10))
thresholds = np.linspace(0.02, 0.50, 97)
n_ev = len(y_ev)
nb_all = prev - (1.0 - prev) * (thresholds / (1.0 - thresholds))

for r_i, arm in enumerate(ARMS):
    an = arm["name"]
    pc, ps = resultados[an]["prob_ev_cont"], resultados[an]["prob_ev_score"]
    ax_c, ax_d = axes[r_i, 0], axes[r_i, 1]

    ax_c.plot([0, 1], [0, 1], ":", color="#666666", label="Ideal")
    for lbl, p, mk, ls, col in [
        ("Camada A: Índice Contínuo", pc, "o", "-", "#1d3557"),
        ("Camada B: Escore de Pontos", ps, "s", "--", "#e63946"),
    ]:
        pt, pp = calibration_curve(y_ev, p, n_bins=10, strategy="quantile")
        ax_c.plot(pp, pt, marker=mk, ls=ls, lw=2, ms=5, color=col, label=f"{lbl} (Brier={brier_score_loss(y_ev, p):.3f})")
    ax_c.set_title(f"Calibração — {arm['label']}")
    ax_c.set_xlabel("Probabilidade Predita Média"); ax_c.set_ylabel("Positividade Observada")
    ax_c.legend(loc="upper left", frameon=True, facecolor="white"); ax_c.grid(alpha=0.25)

    ax_d.plot(thresholds, nb_all, "--", color="#555555", lw=1.2, label="Cultivar todos")
    ax_d.axhline(0.0, ls=":", color="#888888", lw=1.2, label="Cultivar nenhum")
    for lbl, p, ls, col in [
        ("Camada A: Índice Contínuo", pc, "-", "#1d3557"),
        ("Camada B: Escore de Pontos", ps, "--", "#e63946"),
    ]:
        nbs = [(np.sum((p >= pt) & (y_ev == 1)) / n_ev) - (np.sum((p >= pt) & (y_ev == 0)) / n_ev) * (pt / (1.0 - pt)) for pt in thresholds]
        ax_d.plot(thresholds, nbs, ls=ls, lw=2.2, color=col, label=lbl)
    ax_d.set_title(f"Curva de Decisão (DCA) — {arm['label']}")
    ax_d.set_xlabel("Limiar de Probabilidade"); ax_d.set_ylabel("Benefício Líquido")
    ax_d.set_ylim(-0.02, prev + 0.03)
    ax_d.legend(loc="upper right", frameon=True, facecolor="white"); ax_d.grid(alpha=0.25)

fig.suptitle("Calibração e Utilidade Clínica (DCA) — Teste Temporal 2023–2024", fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / "02_Calibracao_e_DCA_Indice_vs_Escore_2023_2024.png", bbox_inches="tight")
plt.show()

# 3. Curva de Risco por Pontuação Total (Predito vs. Observado em 2023–2024)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))
for ax, arm in zip(axes, ARMS):
    lk = resultados[arm["name"]]["lookup_df"]
    lk_obs = lk[lk["n_teste_2023_2024"] >= 15]
    ax.plot(lk["pontuacao_total"], lk["probabilidade_predita_pct"], "-", lw=2.2, color="#1d3557", label="Probabilidade predita (%)")
    ax.scatter(lk_obs["pontuacao_total"], lk_obs["positividade_obs_teste_pct"], color="#e63946", s=36, zorder=5, label="Observado 2023–2024 (n ≥ 15)")
    ax.set_title(f"Risco por Pontuação — {arm['label']}")
    ax.set_xlabel("Pontuação Total do Escore (pontos)")
    ax.set_ylabel("Probabilidade de Urocultura Positiva (%)")
    ax.legend(loc="upper left", frameon=True, facecolor="white"); ax.grid(alpha=0.25)

fig.suptitle("Calibração Direta do Escore de Pontos Inteiros (Teste Temporal 2023–2024)", fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "03_Risco_Por_Pontuacao_Escore_2023_2024.png", bbox_inches="tight")
plt.show()

log(f"Pipeline concluído! Planilha Excel, JSONs e figuras (300 DPI) salvos em: {OUT_DIR}")
